# Drex 2.0 cookbook: a support-triage agent that knows when to ask a human

Drex is not a chat model. You send it a `state` (any text or JSON) and a set of typed questions, and it returns a **probability for every option**. Nothing is generated.

That makes it a good fit for the *decision* step of an agent: route this ticket, flag this refund, escalate or not. In this notebook we build a small support-triage agent on top of the hosted Drex API and measure it on a labeled set:

1. First call
2. The three question types (`noul`, `choice`, `score`) in one request
3. Why batching questions matters (tokens = cost)
4. Turning probabilities into actions: a confidence gate with a human-review lane
5. Evaluation on 40 labeled tickets: accuracy, coverage vs. accuracy, latency, cost
6. Where it fails

Everything runs against the hosted API. No GPU needed.

> Disclosure: this cookbook is part of a sponsored collaboration with Nace AI. The code, dataset and conclusions are my own.

## 0. Setup

Create an API key at [console.nace.ai](https://console.nace.ai) (new accounts get $5 of balance, which is far more than this notebook uses) and expose it as `NACE_API_KEY`, either in your shell or in a `.env` file next to this notebook.

In [ ]:
import os, time, json
from concurrent.futures import ThreadPoolExecutor

import httpx
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.getenv("DREX_BASE_URL", "https://console.nace.ai")
API_KEY = os.environ["NACE_API_KEY"]
MODEL = "drex-v2.0"
PRICE_PER_INPUT_TOKEN = 0.06 / 1_000_000  # USD, drex-v2.0. Output tokens are not billed.

client = httpx.Client(
    base_url=BASE_URL,
    headers={"Authorization": f"Bearer {API_KEY}"},
    timeout=60,  # Drex waits up to 55 s for the model before returning 529
)

The whole API is one endpoint: `POST /v1/systemone`. This helper adds two things: wall-clock timing, and a retry on `429`/`529` that honors `retry-after-ms`, as the docs recommend.

In [ ]:
def decide(state, questions, model=MODEL, max_retries=3):
    body = {"model": model, "state": state, "questions": questions}
    for attempt in range(max_retries + 1):
        t0 = time.perf_counter()
        r = client.post("/v1/systemone", json=body)
        wall_ms = (time.perf_counter() - t0) * 1000
        if r.status_code in (429, 529) and attempt < max_retries:
            time.sleep(int(r.headers.get("retry-after-ms", 1000)) / 1000)
            continue
        if r.status_code != 200:
            raise RuntimeError(f"{r.status_code}: {r.text}")
        out = r.json()
        out["wall_ms"] = wall_ms
        return out

## 1. First call

In [ ]:
res = decide(
    "Help! My payouts have been failing for 3 days.",
    {"is_urgent": {"type": "noul", "instructions": "Does this convey urgency?"}},
)
print(json.dumps(res, indent=2))

`noul` is the probability of **yes**. There is no built-in cutoff: you choose the threshold. `usage.input_tokens` is what you pay for, and `evaluation_time_ms` is the model time on Nace's side (`wall_ms` is what we measured from here, network included).

## 2. Three question types, one request

Our triage agent needs three decisions per ticket:

| Question | Type | Why this type |
|---|---|---|
| `team`: which team handles it | `choice` | One label from an unordered set |
| `wants_refund`: is the customer asking for money back | `noul` | Yes/no with a threshold we control |
| `urgency`: how urgent | `score` | Ordered scale, so we get a fractional position |

In [ ]:
TRIAGE = {
    "team": {
        "type": "choice",
        "instructions": "Which team should handle this support ticket?",
        "criteria": {
            "billing": "Payments, invoices, charges, refunds, plans and pricing",
            "technical": "Bugs, errors, outages, performance and integrations",
            "account": "Login, access, users, permissions and account settings",
            "other": "Anything that is not a support request for the teams above",
        },
    },
    "wants_refund": {
        "type": "noul",
        "instructions": "Is the customer explicitly asking for money back right now?",
    },
    "urgency": {
        "type": "score",
        "instructions": "How urgent is this ticket for the business?",
        "criteria": ["Routine", "Soon", "Urgent"],
    },
}

ticket = "I was charged twice for my October invoice. Please refund the duplicate charge."
res = decide(ticket, TRIAGE)
print(json.dumps(res["answers"], indent=2))
print("input tokens:", res["usage"]["input_tokens"], "| model ms:", res.get("evaluation_time_ms"))

- `choice` returns the top label **and** the full distribution. If the top two are close, the model is telling you it can't separate them.
- `score` returns the *expected* level (sum of index × probability), so `1.4` means "between Soon and Urgent".

## 3. Batch your questions

Drex bills input tokens only. One request with three questions sends the ticket once; three requests send it three times.

In [ ]:
batched = decide(ticket, TRIAGE)["usage"]["input_tokens"]
separate = sum(decide(ticket, {k: q})["usage"]["input_tokens"] for k, q in TRIAGE.items())
print(f"batched: {batched} tokens | separate: {separate} tokens | saved: {1 - batched / separate:.0%}")

## 4. From probabilities to actions

A probability is only useful if your code does something with it. The agent below acts on its own when Drex is confident and sends everything else to a person:

- **Route automatically** if the top team probability is at least `team_min`.
- **Start the refund flow** if `wants_refund ≥ refund_hi`; ignore refunds below `refund_lo`; anything in between goes to a human.
- **Page on-call** if the expected urgency is at least `page_at`.

The thresholds are business decisions, not model properties. Log the raw probabilities so you can move them later.

In [ ]:
def triage(ans, team_min=0.85, refund_hi=0.9, refund_lo=0.1, page_at=1.5):
    team = ans["team"]["choice"]
    p_team = ans["team"]["probabilities"][team]
    p_refund = ans["wants_refund"]["noul"]
    urgency = ans["urgency"]["score"]

    actions = []
    if p_team < team_min:
        return ["human_review: unsure about team"]
    actions.append(f"route:{team}")
    if p_refund >= refund_hi:
        actions.append("start_refund_flow")
    elif p_refund > refund_lo:
        return actions + ["human_review: unsure about refund"]
    if urgency >= page_at:
        actions.append("page_on_call")
    return actions

for text in [
    ticket,
    "SSO login is broken for our whole company since 9am, 200 people can't work.",
    "It's not working. Please fix.",
]:
    print(f"{text[:60]:<62} -> {triage(decide(text, TRIAGE)['answers'])}")

## 5. Evaluation on 40 labeled tickets

`data/tickets.jsonl` holds 40 synthetic tickets I wrote and labeled by hand. 20 are clear-cut; the rest are deliberately hard: multiple intents, sarcasm, Spanish, negations ("I don't want a refund"), vague one-liners and prompt-injection attempts.

It's a small set, so treat the numbers as a sanity check of the workflow, not a benchmark. Free accounts allow 8 requests in flight, so we cap concurrency there.

In [ ]:
tickets = pd.read_json("data/tickets.jsonl", lines=True)
URGENCY = ["Routine", "Soon", "Urgent"]

def run(row):
    res = decide(row.text, TRIAGE)
    a = res["answers"]
    return {
        "id": row.id,
        "pred_team": a["team"]["choice"],
        "p_team": a["team"]["probabilities"][a["team"]["choice"]],
        "p_refund": a["wants_refund"]["noul"],
        "urgency_score": a["urgency"]["score"],
        "input_tokens": res["usage"]["input_tokens"],
        "model_ms": res.get("evaluation_time_ms"),
        "wall_ms": res["wall_ms"],
        "answers": a,
    }

with ThreadPoolExecutor(max_workers=8) as pool:
    preds = list(pool.map(run, tickets.itertuples()))

df = tickets.merge(pd.DataFrame(preds), on="id")
df["team_ok"] = df.pred_team == df.team
df["refund_ok"] = (df.p_refund >= 0.5) == df.wants_refund
df["urgency_err"] = (df.urgency_score - df.urgency).abs()

os.makedirs("results", exist_ok=True)
df.drop(columns="answers").to_csv("results/eval.csv", index=False)
df[["id", "tag", "team", "pred_team", "p_team", "wants_refund", "p_refund", "urgency", "urgency_score"]].round(3)

### Accuracy, overall and by ticket type

In [ ]:
summary = pd.DataFrame({
    "team accuracy": df.groupby("tag").team_ok.mean(),
    "refund accuracy (@0.5)": df.groupby("tag").refund_ok.mean(),
    "urgency MAE": df.groupby("tag").urgency_err.mean(),
    "n": df.groupby("tag").size(),
})
summary.loc["ALL"] = [df.team_ok.mean(), df.refund_ok.mean(), df.urgency_err.mean(), len(df)]
summary.round(2)

### Coverage vs. accuracy

This is the chart that matters for an agent. Raising the confidence threshold sends fewer tickets through automatically (coverage goes down). If the probabilities are well calibrated, the tickets that do go through should get *more* accurate. If the accuracy line stays flat, the confidence isn't telling you anything.

In [ ]:
rows = []
for t in [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 0.98, 0.99]:
    auto = df[df.p_team >= t]
    rows.append({"threshold": t, "coverage": len(auto) / len(df),
                 "accuracy_on_auto": auto.team_ok.mean() if len(auto) else float("nan")})
curve = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(curve.threshold, curve.coverage, marker="o", label="coverage (auto-routed)")
ax.plot(curve.threshold, curve.accuracy_on_auto, marker="o", label="accuracy on auto-routed")
ax.set_xlabel("team confidence threshold"); ax.set_ylim(0, 1.05); ax.grid(alpha=0.3); ax.legend()
ax.set_title("Team routing: coverage vs. accuracy")
plt.tight_layout(); plt.savefig("results/coverage_accuracy.png", dpi=150); plt.show()
curve.round(3)

### What the agent would actually do

In [ ]:
df["actions"] = df.answers.apply(triage)
df["auto"] = ~df.actions.apply(lambda a: any(x.startswith("human_review") for x in a))
auto = df[df.auto]
print(f"handled automatically: {df.auto.mean():.0%} ({df.auto.sum()}/{len(df)})")
print(f"team accuracy on automatic: {auto.team_ok.mean():.0%}")
print(f"refund accuracy on automatic: {auto.refund_ok.mean():.0%}")
df[["id", "tag", "text", "actions"]]

### Latency and cost

`model_ms` is Drex's own `evaluation_time_ms`; `wall_ms` includes the network round trip from wherever you run this notebook (and queueing, since we sent 8 at a time).

In [ ]:
lat = df[["model_ms", "wall_ms"]].describe(percentiles=[0.5, 0.95]).loc[["50%", "95%", "max"]]
print(lat.round(1))
tokens = df.input_tokens.sum()
print(f"\n{len(df)} tickets x 3 questions = {tokens} input tokens = ${tokens * PRICE_PER_INPUT_TOKEN:.6f}")
print(f"per 1M tickets at this size: ${tokens / len(df) * 1_000_000 * PRICE_PER_INPUT_TOKEN:,.2f}")

## 6. Where it fails

Every mistake, with the probabilities behind it. The interesting question isn't *whether* it fails but whether it fails **with low confidence**, because then the gate in section 4 catches it.

In [ ]:
errors = df[~df.team_ok | ~df.refund_ok]
errors[["id", "tag", "text", "team", "pred_team", "p_team", "wants_refund", "p_refund", "auto"]].round(3)

### Prompt injection

Drex can't follow instructions in the ticket the way a chat model might, because it only scores the options you defined. It can still be *influenced* by them. These are the two injection tickets:

In [ ]:
df[df.tag == "injection"][["id", "text", "pred_team", "p_team", "p_refund", "urgency_score", "actions"]].round(3)

## 7. Takeaways

Fill this in from your own run; the numbers above are what you'll quote. The questions to answer:

- How much of the traffic did the gate handle on its own, and how accurate was that slice?
- Did higher confidence actually mean higher accuracy (section 5 chart)?
- Which ticket types broke it, and did they break with low or high confidence?
- What did latency and cost look like at this volume?

Next steps: tune the thresholds on real traffic, log raw probabilities, and replay past tickets when you change them.